In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

d:\Anaconda\envs\pytorch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_data = pd.read_csv("samsum-train.csv")
test_data = pd.read_csv("samsum-test.csv")
validation_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.sample(10)

,id,dialogue,summary
6726,13681341,Andy: Suh?\r\nLuke: Not much. S'up?\r\nAndy: Y...,Luke will visit Andy to play the new Assassin'...
7734,13810091,"Austin: Hey you, put channel 8 on\r\nDarcy: wh...",Austin tells Darcy to put Channel 8 on as they...
2910,13813947,Lyric: How old do you want to live to?\r\nColt...,Colt wants to live up to 1000 years and Lyric ...
12845,13729305,Ginny: what is going on with this bloody dog p...,Ginny is furious about people not cleaning up ...
13079,13681583,Lydia: are you back from Barcelona?\r\nRoger: ...,Roger's flight from Barcelona has just touched...
4361,13829963,Dan: late breakfast tomorrow?\r\nPatrick: soun...,Dan and Patrick will have a late breakfast tom...
4342,13681612,Lilyanna: Morning\r\nErnest: Good morning\r\nL...,Lilyanna and Ernest have exams to take tomorrow.
13578,13716587,Ola: What's up in your cart? Ours is horrible ...,Ola's cart is horrible and Mateusz's is normal...
6348,13728334-1,"Ella: Hey, u have a dog, right?\r\nCecil: Yup....",Ella wants Cecil's recommendation on where to ...
12687,13828217,Henrik: guess what your mother textd me\r\nEmi...,Emily's mother sent Henrik photos of Emily as ...


In [4]:
import re
def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # line character
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags <p>, <h1>
    text = text.strip().lower()
    return text

In [5]:
train_data[train_data["dialogue"].apply(lambda x: isinstance(x, float))]

,id,dialogue,summary
6054,13828807,NaN,problem with visualization of the content


In [6]:
train_data = train_data.dropna(subset=["dialogue"])

In [7]:
# train_data["dialogue"].apply(                    # isinstance function can be used to check datatype 
#     lambda x: x if isinstance(x, float) else ""
# )

In [8]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

validation_data["dialogue"] = validation_data["dialogue"].apply(clean_data)
validation_data["summary"] = validation_data["summary"].apply(clean_data)

### Tokenize

In [9]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [10]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding = "max_length", max_length = 512, truncation = True)
    targets = tokenizer(data["summary"], padding = "max_length", max_length = 150, truncation = True)

    inputs["labels"] = targets["input_ids"]
    return inputs

In [11]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist()
val_dataset = validation_data.apply(tokenize, axis = 1).tolist()

In [12]:
train_dataset[0]

{'input_ids': [183, 232, 9, 10, 3, 23, 13635, 5081, 5, 103, 25, 241, 128, 58, 3, 12488, 651, 10, 417, 55, 183, 232, 9, 10, 3, 23, 31, 195, 830, 25, 5721, 3, 10, 18, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

### Finetuning

In [13]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 3805.00it/s]


In [14]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [15]:
# Training Arguments

training_args =  TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 6,
    weight_decay = 0.01,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    eval_strategy= "epoch",
    save_strategy= "epoch",
    warmup_steps= 500 # means in 500 steps learning rate will go from 0 to defualt value (learning is slow at start and fast as we move furthur)
)

In [16]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset =  train_dataset,
    eval_dataset = val_dataset
)

In [17]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.385283,0.340674
2,0.363162,0.330010
3,0.340956,0.324579
4,0.346644,0.323501
5,0.340976,0.321354
6,0.338800,0.321160


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.59it/s]


TrainOutput(global_step=11052, training_loss=0.5014787193143708, metrics={'train_runtime': 7059.7454, 'train_samples_per_second': 12.52, 'train_steps_per_second': 1.565, 'total_flos': 1.1962320464904192e+16, 'train_loss': 0.5014787193143708, 'epoch': 6.0})

In [18]:
model.save_pretrained('./saved_summary_model')
tokenizer.save_pretrained('./saved_summary_model')
# to load the same use from_pretrained() instead of save_pretrained()

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.17it/s]


('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [19]:
model = T5ForConditionalGeneration.from_pretrained('./saved_summary_model')
tokenizer = T5Tokenizer.from_pretrained('./saved_summary_model')

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 3818.38it/s]


In [24]:
def summarized_dialogue(dialogue):
    # clean
    dialogue = clean_data(dialogue)

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding = 'max_length',
        max_length = 512,
        truncation = True,
        return_tensors = 'pt'
    ).to(device)
    # generate summary ==> token ids
    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams = 4,     # generate 4 summaries for each dialogue and the one which is best is chosen as output
        early_stopping = True  # stops at EOS
    )
    # token ids convert to summary ==> decoding 
    summary = tokenizer.decode(targets[0], skip_special_token = True)
    return summary

In [28]:
text = """

**Sarah:** Alright everyone, let's get started. We're already two weeks into the sprint and I want to know where we stand on the dashboard feature.

**Raj:** The backend API is mostly done. I finished the authentication endpoints yesterday, but I'm still working on the data aggregation logic. Probably another two days on that.

**Emily:** On the frontend side, the UI components are built and connected to the mock data. I'm just waiting on Raj's API to go live so I can swap in the real endpoints. Shouldn't take long once that's ready.

**Sarah:** Good. What about the export to PDF feature? That was supposed to be part of this sprint too.

**Raj:** Yeah, I haven't touched that yet. Honestly, I think we might need to push it to next sprint. The aggregation logic is more complex than I estimated.

**Emily:** I agree. Better to do it right than rush it. The core dashboard is more important for the demo anyway.

**Sarah:** Okay, fair enough. Let's formally move the PDF export to Sprint 4. I'll update the board. One last thing — the client demo is on Friday at 3 PM. Can both of you be available?

**Raj:** Works for me.

**Emily:** I have a conflict until 2:30, but I'll be there by 3.

**Sarah:** Perfect. Let's aim to have everything merged and deployed to staging by Thursday EOD so we have time to test. Any blockers I should know about?

**Raj:** Nothing major. I might need a code review from Emily on the aggregation logic tomorrow.

**Emily:** Sure, just send me the PR.

**Sarah:** Great. That's all for now — thanks everyone.

"""
summary = summarized_dialogue(text)
summary = summary.replace("<pad>", "").replace("</s>", "").strip()
print(summary)

raj finished the authentication endpoints yesterday, but he's still working on the data aggregation logic. raj will move the pdf export to sprint 4. sarah will update the client demo on friday at 3 pm.
